# PlantDoctor AI 🌿 — Final Submission Notebook

**Plant leaf disease classification with traditional machine learning**  
**Dataset:** PlantVillage (`emmarex/plantdisease`) · **15 classes** · **Final classifier:** LightGBM  
**Held-out test:** **95.1842% accuracy**, **93.8936% macro F1**, **3,094 samples**.

> **Submission / reproducibility note:** This is a cleaned, non-destructive reconstruction of the original experiment notebook. The trained model and extracted feature arrays are **external artifacts**, not embedded in this notebook. Restore them from the saved Kaggle outputs / backups before running. The default path **loads and reports frozen artifacts**; costly feature extraction, retraining, and re-evaluation of the test set are disabled by default. Historical experiment scores below were recorded in the original notebook, not recomputed here. Do not claim a new independent test run from simply viewing saved results.

## 1. Environment and artifact locations

Attach the PlantVillage Kaggle dataset. Place the following in `/kaggle/working/plantdoctor/` (from the saved backups): `all_features.npy`, `all_metadata.csv`, `split_indices.npz`, `label_encoder.joblib`, `lgbm_selected_features.npy`, `lightgbm_850.joblib`, and the `final_evaluation/` files. These paths can be changed below.

**Never unpickle untrusted `.joblib` files.**

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
DATA_DIR = Path('/kaggle/input/datasets/emmarex/plantdisease/plantvillage/PlantVillage')
OUT = Path('/kaggle/working/plantdoctor')
REPORTS = OUT / 'final_evaluation'
OUT.mkdir(parents=True, exist_ok=True)

print('Dataset mounted:', DATA_DIR.exists())
print('Artifact directory:', OUT)
for name in ['all_features.npy', 'all_metadata.csv', 'split_indices.npz',
             'label_encoder.joblib', 'lgbm_selected_features.npy',
             'lightgbm_850.joblib', 'final_evaluation/test_metrics.json']:
    print(('OK     ' if (OUT / name).exists() else 'MISSING'), name)

## 2. Dataset exploration

The source notebook selected a single PlantVillage directory to avoid duplicate copies of the dataset. The original feature matrix contains **20,638 image rows**; **14 identical feature vectors** were excluded from the split, leaving **20,624 unique feature rows**. Counts reflect this project's dataset copy and preprocessing, not all PlantVillage releases.

In [ ]:
if DATA_DIR.exists():
    records = [
        {'path': str(p), 'class': class_dir.name}
        for class_dir in sorted(DATA_DIR.iterdir()) if class_dir.is_dir()
        for p in sorted(class_dir.iterdir())
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    ]
    dataset_df = pd.DataFrame(records)
    print('Images on disk:', len(dataset_df))
    print('Classes:', dataset_df['class'].nunique())
    display(dataset_df['class'].value_counts().rename('images').to_frame())
else:
    print('Dataset not mounted; attach emmarex/plantdisease to browse raw images.')

## 3. Exact image feature extractor

**Critical:** Use OpenCV's default `cv2.resize(img, (64, 64))` (**INTER_LINEAR**). An abandoned experiment used `INTER_AREA`, which is **not** the pipeline used to generate the saved model's input features. The code below is extracted from the original feature-extraction function. It yields **1,632** features: 48 HSV histogram values, 6 RGB statistics, 10 uniform LBP histogram values, and 1,568 HOG values. The deployment extractor was checked against 20 saved training images, all matched exactly at the reported precision.

In [ ]:
import cv2
from skimage.feature import hog, local_binary_pattern

def extract_features(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f'Cannot read: {path}')

    img = cv2.resize(img, (64, 64))  # default INTER_LINEAR: DO NOT CHANGE
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    features = []
    for channel, maximum in [(0, 180), (1, 256), (2, 256)]:
        hist = cv2.calcHist([hsv], [channel], None, [16], [0, maximum]).ravel()
        hist /= hist.sum() + 1e-8
        features.extend(hist)

    for channel in range(3):
        values = rgb[:, :, channel]
        features.extend([values.mean() / 255, values.std() / 255])

    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(11))
    hist = hist / (hist.sum() + 1e-8)
    features.extend(hist)

    h = hog(gray, orientations=8, pixels_per_cell=(8, 8),
            cells_per_block=(2, 2), block_norm='L2-Hys')
    features.extend(h)

    result = np.asarray(features, dtype=np.float32)
    assert result.shape == (1632,), result.shape
    return result

## 4. Load the cached feature matrix and original splits

Feature extraction originally took substantial time. This final notebook **loads** the existing arrays; it does not recompute 20,638 images by default. The saved split indices reference **rows of the original 20,638-row matrix**. Do not reset or reindex the feature matrix independently of `all_metadata.csv`.

The original split used stratification and random seed 42, with **14,436 train / 3,094 validation / 3,094 test** samples. The test set was held out from model selection.

In [ ]:
required = ['all_features.npy', 'all_metadata.csv', 'split_indices.npz',
            'label_encoder.joblib', 'lgbm_selected_features.npy',
            'lightgbm_850.joblib']
missing = [f for f in required if not (OUT / f).is_file()]
if missing:
    raise FileNotFoundError(
        'Restore these files from your saved Kaggle backups before continuing: '
        + ', '.join(missing)
    )

X = np.load(OUT / 'all_features.npy', mmap_mode='r')
metadata = pd.read_csv(OUT / 'all_metadata.csv')
splits = np.load(OUT / 'split_indices.npz')
encoder = joblib.load(OUT / 'label_encoder.joblib')
selected = np.load(OUT / 'lgbm_selected_features.npy')
model = joblib.load(OUT / 'lightgbm_850.joblib')

train_idx, val_idx, test_idx = (splits[k] for k in ('train_idx', 'val_idx', 'test_idx'))
assert X.shape == (20638, 1632), X.shape
assert len(metadata) == len(X)
assert len(selected) == 350 and len(set(selected.tolist())) == 350
assert selected.min() >= 0 and selected.max() < X.shape[1]
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert (len(train_idx), len(val_idx), len(test_idx)) == (14436, 3094, 3094)
assert np.array_equal(model.classes_, np.arange(len(encoder.classes_)))

print('Raw feature matrix:', X.shape)
print('Selected features:', len(selected))
print('Train / validation / test:', len(train_idx), len(val_idx), len(test_idx))
print('Classes:', len(encoder.classes_))

### Optional: verify extractor alignment on training images

This is a **read-only** check. It uses training rows, not the test set. If original image paths no longer exist, skip this optional cell.

In [ ]:
if DATA_DIR.exists():
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(train_idx, size=min(20, len(train_idx)), replace=False)
    diffs = []
    for i in sample_idx:
        path = metadata.iloc[int(i)]['path']
        if not Path(path).is_file():
            raise FileNotFoundError(f'Original image unavailable: {path}')
        rebuilt = extract_features(path)
        saved = np.asarray(X[int(i)], dtype=np.float32)
        diff = float(np.max(np.abs(rebuilt - saved)))
        assert np.allclose(rebuilt, saved, rtol=1e-5, atol=1e-6), (i, diff)
        diffs.append(diff)
    print(f'Feature verification: {len(diffs)}/{len(sample_idx)} matched; max diff={max(diffs):.10f}')
else:
    print('Skipped: source images are not mounted.')

## 5. Feature selection and model development

The final 350 features were selected using `feature_importances_` from the **best ExtraTrees model**, trained using the training split. The original source notebook saved the indices as `lgbm_selected_features.npy`. The cell below documents the **exact historical feature-selection rule** without changing the saved indices.

The selected LightGBM model was trained using 850 estimators, learning rate 0.05, 31 leaves, `min_child_samples=20`, column subsampling 0.85, row subsampling 0.8, `reg_lambda=3`, `random_state=42`, and `n_jobs=2`. An internal training-only early-stopping split was used to choose the boosting round count; it reached 850 without triggering early stopping.

In [ ]:
# Documentation / verification only; do not recompute feature selection.
print('Selected indices:', selected.shape)
print('Final model:', type(model).__name__)
print('Model parameters (selected):')
for k in ['n_estimators', 'learning_rate', 'num_leaves',
          'min_child_samples', 'colsample_bytree', 'subsample',
          'subsample_freq', 'reg_lambda', 'random_state']:
    print(f'  {k}: {model.get_params().get(k)}')

# To reproduce selection FROM SCRATCH, retrain the historical best ExtraTrees
# configuration (150 estimators, max_features=.30, min_samples_leaf=2,
# class_weight="balanced", random_state=42), then apply:
# selected = np.argsort(best_extratrees.feature_importances_)[-350:]
# Do NOT overwrite the existing saved indices for the submitted model.

### Optional: original final training recipe (disabled)

This cell is preserved for methodological reproducibility. **It is disabled by default** because the model is already trained and tested. Running it again may not reproduce the exact serialized artifact under different library versions. It must never overwrite `lightgbm_850.joblib` used for the reported test results.

In [ ]:
RUN_RETRAINING = False  # Set True only in a separate research copy, not for submission.
if RUN_RETRAINING:
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score

    y_train = encoder.transform(metadata.iloc[train_idx]['class'])
    y_val = encoder.transform(metadata.iloc[val_idx]['class'])
    X_train = np.asarray(X[train_idx][:, selected], dtype=np.float32)
    X_val = np.asarray(X[val_idx][:, selected], dtype=np.float32)

    fit_idx, stop_idx = train_test_split(
        np.arange(len(y_train)), test_size=0.15,
        stratify=y_train, random_state=42
    )
    params = dict(
        objective='multiclass', num_class=len(encoder.classes_),
        n_estimators=850, learning_rate=0.05, num_leaves=31,
        max_depth=-1, min_child_samples=20, colsample_bytree=0.85,
        subsample=0.8, subsample_freq=1, reg_lambda=3.0,
        n_jobs=2, random_state=42, verbosity=-1
    )
    pilot = lgb.LGBMClassifier(**params)
    pilot.fit(X_train[fit_idx], y_train[fit_idx],
              eval_set=[(X_train[stop_idx], y_train[stop_idx])],
              eval_metric='multi_logloss',
              callbacks=[lgb.early_stopping(60, verbose=True)])
    rounds = int(pilot.best_iteration_ or 850)
    candidate = lgb.LGBMClassifier(**{**params, 'n_estimators': rounds})
    candidate.fit(X_train, y_train)
    pred_val = candidate.predict(X_val)
    print('Research validation accuracy:', accuracy_score(y_val, pred_val))
    print('Research validation macro F1:', f1_score(y_val, pred_val, average='macro'))
    print('Candidate not saved: frozen deployment model remains unchanged.')
else:
    print('Retraining intentionally skipped; using frozen LightGBM artifact.')

## 6. Model comparison — historical validation results

These values are transcribed from the **original notebook outputs** and were obtained on the **validation split**, not on the held-out test split. The experiments included SGD, ExtraTrees, LightGBM, and ensembles. The 50/50 LightGBM–ExtraTrees blend had slightly higher validation macro F1, but the **single LightGBM 850** was chosen for a simpler and faster deployment with nearly identical performance.

In [ ]:
validation_results = pd.DataFrame([
    {'Model': 'SGD baseline', 'Accuracy': 0.7679, 'Macro F1': 0.7411},
    {'Model': 'ExtraTrees baseline', 'Accuracy': 0.904654, 'Macro F1': 0.897245},
    {'Model': 'ExtraTrees best (150)', 'Accuracy': 0.924370, 'Macro F1': 0.918460},
    {'Model': 'LightGBM 350', 'Accuracy': 0.949257, 'Macro F1': 0.945185},
    {'Model': 'LightGBM 850 — DEPLOYED', 'Accuracy': 0.952165, 'Macro F1': 0.948494},
    {'Model': 'LGB 850 + ExtraTrees 50/50', 'Accuracy': 0.951842, 'Macro F1': 0.949094},
])
display(validation_results.style.format({'Accuracy': '{:.2%}', 'Macro F1': '{:.2%}'}))

ax = validation_results.set_index('Model')[['Accuracy', 'Macro F1']].plot.barh(figsize=(9, 4.5))
ax.set_xlabel('Validation score')
ax.set_xlim(0.7, 1.0)
ax.set_title('Historical validation comparison (not test results)')
plt.tight_layout()
plt.show()

### Feature selection ablation (historical)

At 300 LightGBM estimators, using the **selected 350 features** achieved validation accuracy **94.6671%** and macro F1 **94.3177%** in approximately **106 seconds**; all 1,632 features achieved **94.2793%** and **93.6643%** in approximately **530 seconds**. These timings are environment-dependent. The result supported retaining feature selection.

## 7. Independent held-out test — final recorded results

**The test set was evaluated once after model selection.** To preserve that separation, this notebook **reads the already saved final metrics and reports** rather than running a new test evaluation. The scores below are the recorded results from the original notebook. Do not tune any model based on the test set.

| Metric | Final held-out test |
|---|---:|
| Accuracy | **95.1842%** |
| Macro F1 | **93.8936%** |
| Macro Precision | **94.4751%** |
| Macro Recall | **93.3954%** |
| Samples | **3,094** |
| Correct / incorrect | **2,945 / 149** |

In [ ]:
metrics_path = REPORTS / 'test_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Recorded final independent test metrics:')
    display(pd.DataFrame([metrics]).T.rename(columns={0: 'value'}))
    assert abs(metrics['accuracy'] - 2945 / 3094) < 1e-10
    assert metrics['test_samples'] == 3094
else:
    print('Recorded summary: Accuracy 95.1842%; Macro F1 93.8936%; '
          'Macro Precision 94.4751%; Macro Recall 93.3954%; 2945/3094 correct.')
    print('Restore final_evaluation/test_metrics.json for the source report.')

report_path = REPORTS / 'classification_report.csv'
if report_path.exists():
    report = pd.read_csv(report_path, index_col=0)
    display(report)
else:
    print('Detailed classification report not found in the current artifact folder.')

cm_path = REPORTS / 'confusion_matrix.csv'
if cm_path.exists():
    cm = pd.read_csv(cm_path, index_col=0)
    print('Confusion matrix:', cm.shape)
    display(cm)

## 8. Inference with the frozen model (validation example)

This demonstration deliberately uses **validation**, not test, and does not modify any trained artifact. In the deployed Streamlit app, the exact same extractor is applied to uploaded images before selecting the 350 feature indices. Probabilities are **not calibrated**, and a high model probability does not guarantee a correct diagnosis.

In [ ]:
idx = int(val_idx[0])
features = np.asarray(X[idx], dtype=np.float32)[selected].reshape(1, -1)
proba = model.predict_proba(features)[0]
order = np.argsort(proba)[::-1][:3]
print('Example validation image:', metadata.iloc[idx]['path'])
print('Reference label:', metadata.iloc[idx]['class'])
for j in order:
    print(f'{encoder.classes_[j]}: {proba[j]:.2%}')

## 9. Deployment and limitations

The Streamlit application is hosted from the [PlantDoctor-AI GitHub repository](https://github.com/nada-ali-13/PlantDoctor-AI). The deployed app uses `lightgbm_850.joblib`, `lgbm_selected_features.npy`, `label_encoder.joblib`, and the verified `feature_extractor.py`. The full image dataset and training feature matrix are **not required** for online inference.

**Limitations:** The evaluation uses held-out PlantVillage images, not a representative field-photo benchmark. The classifier always selects one of its 15 known classes and does not reliably reject unsupported plants or unrelated photos. Its class probabilities are not statistically calibrated. In an external example, a potato late-blight image was predicted as tomato late blight (the disease matched, but the combined class was wrong). A future optional plant-type selector may help when the user knows the plant type; it has not been implemented or validated.

**Conclusion:** The traditional feature-engineering + LightGBM pipeline achieved **95.18%** held-out accuracy and **93.89%** macro F1 across 15 classes, with a verified extractor and a working Streamlit interface. These results support the project as a proof of concept, not a substitute for expert agricultural diagnosis.